# Macro Dashboard: Visualizing the U.S. Economy

We typically look at macroeconomic data one at a time, whether it's a headline inflation print, a monthly jobs report or a rate decision. However, the economic system moves together, so individually analyzing each figure leaves out the big picture and the insights that may come from it.

In this notebook, we look at four core indicators from the [FRED API](https://fred.stlouisfed.org/) and visualize how they relate to one another over time.

| Indicator | FRED series | Frequency | Role in the system |
|---|---|---|---|
| Real GDP growth | `A191RL1Q225SBEA` | Quarterly | Baseline measure of expansion vs. contraction |
| CPI inflation | `CPIAUCSL` (index → YoY %) | Monthly | Price-stability half of the Fed's dual mandate |
| Unemployment rate | `UNRATE` | Monthly | Labor-market half of the mandate |
| Federal funds rate | `FEDFUNDS` | Monthly | The Fed's policy lever |

We also grab `USREC` which is the NBER recession indicator in order to shade any times of recession on our charts, as this is a key factor in how the economy and Fed act.



# 1. Setup

First, we must get our free FRED API key to pull our data. The key is read from the `FRED_API_KEY` environment variable so it never appears in the notebook or the repo.

In [22]:
import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from fredapi import Fred
from dotenv import load_dotenv
load_dotenv()
API_KEY = os.environ.get("FRED_API_KEY")
assert API_KEY, "Set the FRED_API_KEY environment variable (or put it in a .env file)."

fred = Fred(api_key=API_KEY)

# 2. Bulding the Pipeline

## 2.1 Pulling Raw Series

Our 4 indicators come in different units and frequencies. GDP growth is already a quarterly, seasonally-adjusted annualized rate; CPI is a monthly *index level* that has to be converted into an inflation rate; unemployment and the funds rate are monthly percentages that can be used as-is.

The time window I want to use is `1990-01-01` onward, which is long enough to cover the 1990s expansion, the dot-com and 2008 recessions, and the 2021–23 inflation episode, while keeping the charts readable. I will also explore the 1960s–70s, which adds the Great Inflation (useful for the Phillips-curve) at the cost of visual clutter.

In [25]:
START_DATE = "1990-01-01" 

SERIES = {
    "gdp_growth": "A191RL1Q225SBEA",  # Real GDP growth, % chg, quarterly
    "cpi_index":  "CPIAUCSL",         # CPI, index level, monthly
    "unemp":      "UNRATE",           # Unemployment rate, %, monthly
    "fedfunds":   "FEDFUNDS",         # Effective federal funds rate, %, monthly
    "recession":  "USREC",            # NBER recession indicator (0/1), monthly
}

raw = {name: fred.get_series(code, observation_start=START_DATE)
       for name, code in SERIES.items()}

for name, s in raw.items():
    print(f"{name}: {len(s)} obs | {s.index.min().date()} → {s.index.max().date()}")

gdp_growth: 145 obs | 1990-01-01 → 2026-01-01
cpi_index: 437 obs | 1990-01-01 → 2026-05-01
unemp: 438 obs | 1990-01-01 → 2026-06-01
fedfunds: 438 obs | 1990-01-01 → 2026-06-01
recession: 438 obs | 1990-01-01 → 2026-06-01


## 2.2 Transforming the Data

This first thing we need to do is convert our CPI_index to inflation rates, which is calculated as:
$$
\text{Inflation Rate} = \frac{\text{CPI}_{\text{new}} - \text{CPI}_{\text{old}}}{\text{CPI}_{\text{old}}} \times 100
$$
We're looking to get the year to year inflation rate, so I computed `pct_change(12)` on the monthly index.

In [26]:
# CPI index -> year-over-year inflation rate
inflation = raw["cpi_index"].pct_change(12) * 100

# Create a DataFrame with the monthly data
df = pd.DataFrame({
    "inflation": inflation,
    "unemp":     raw["unemp"],
    "fedfunds":  raw["fedfunds"],
    "recession": raw["recession"],
})

C:\Users\tasso\AppData\Local\Temp\ipykernel_9176\3570216203.py:2: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  inflation = raw["cpi_index"].pct_change(12) * 100
